# Import Required Libraries
Import the same libraries used in the training notebook, such as pandas, numpy, matplotlib, seaborn, sklearn, holidays.

In [1]:
# Import libraries for forecasting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import holidays

# Load ventas_2025_inferencias Data
Load the ventas_2025_inferencias file from data/raw/inferencia into a pandas DataFrame named inferencia_df.

In [2]:
# Load ventas_2025_inferencias.csv into inferencia_df
inferencia_df = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')
inferencia_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [3]:
# 1. Cargar inferencia_df
import pandas as pd

inferencia_path = '../data/raw/inferencia/ventas_2025_inferencia.csv'
inferencia_df = pd.read_csv(inferencia_path)

# 2. Asegurar formato de fecha
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])

# 3. Variables temporales básicas
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.dayofweek  # 0=Lunes, 6=Domingo

# 4. Nombre del día de la semana
inferencia_df['nombre_dia_semana'] = inferencia_df['fecha'].dt.day_name(locale='es_ES') if hasattr(inferencia_df['fecha'].dt, 'day_name') else inferencia_df['fecha'].dt.dayofweek.map({0:'Lunes',1:'Martes',2:'Miércoles',3:'Jueves',4:'Viernes',5:'Sábado',6:'Domingo'})

# 5. Es fin de semana
inferencia_df['es_fin_de_semana'] = inferencia_df['dia_semana'].isin([5, 6])

# 6. Festivos en España
import holidays
festivos_es = holidays.country_holidays('ES', years=inferencia_df['año'].unique())
inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(festivos_es)

# 7. Black Friday y Cyber Monday
def black_friday(year):
    nov = pd.date_range(start=f'{year}-11-01', end=f'{year}-11-30', freq='D')
    return nov[nov.weekday == 4][-1]
inferencia_df['es_black_friday'] = inferencia_df['fecha'].apply(lambda x: x == black_friday(x.year))

def cyber_monday(year):
    bf = black_friday(year)
    return bf + pd.Timedelta(days=(7 - bf.weekday()) % 7 + 3) if bf.weekday() <= 4 else bf + pd.Timedelta(days=3)
inferencia_df['es_cyber_monday'] = inferencia_df['fecha'].apply(lambda x: x == cyber_monday(x.year))

# 8. Principio y fin de mes
inferencia_df['es_principio_mes'] = inferencia_df['dia_mes'] <= 3
inferencia_df['es_fin_mes'] = inferencia_df['dia_mes'] >= (inferencia_df['fecha'] + pd.offsets.MonthEnd(0)).dt.day - 2

# 9. Día laborable
inferencia_df['es_laborable'] = ~inferencia_df['es_festivo'] & ~inferencia_df['es_fin_de_semana']

# 10. Día víspera y post festivo
inferencia_df['es_víspera_festivo'] = inferencia_df['fecha'].shift(-1).isin(festivos_es)
inferencia_df['es_post_festivo'] = inferencia_df['fecha'].shift(1).isin(festivos_es)

# 11. Semana del año y trimestre
inferencia_df['semana_año'] = inferencia_df['fecha'].dt.isocalendar().week
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter

# 12. Lags y media móvil (por producto y año)
lag = [1,2,3,4,5,6,7]
for l in lag:
    inferencia_df[f'unidades_vendidas_lag{l}'] = inferencia_df.groupby(['producto_id', 'año'])['unidades_vendidas'].shift(l)
inferencia_df['unidades_vendidas_mm7'] = inferencia_df.groupby(['producto_id', 'año'])['unidades_vendidas'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())

# 13. Codificaciones y variables adicionales (replicar las de df)
# Si tienes variables dummies o codificaciones específicas, añade aquí el mismo proceso que en el notebook de entrenamiento.

# 14. Eliminar registros de octubre y dejar solo noviembre
inferencia_df = inferencia_df[inferencia_df['mes'] == 11]

# 15. Guardar el dataframe transformado
inferencia_df.to_csv('../data/processed/inferencia_df_transformado.csv', index=False)

C:\Users\kuris\AppData\Local\Temp\ipykernel_9488\2901829482.py:25: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(festivos_es)
C:\Users\kuris\AppData\Local\Temp\ipykernel_9488\2901829482.py:46: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  inferencia_df['es_víspera_festivo'] = inferencia_df['fecha'].shift(-1).isin(festivos_es)
C:\Users\kuris\AppData\Local\Temp\ipykernel_9488\2901829482.py:47: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future versi

In [4]:
inferencia_df

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,semana_año,trimestre,unidades_vendidas_lag1,unidades_vendidas_lag2,unidades_vendidas_lag3,unidades_vendidas_lag4,unidades_vendidas_lag5,unidades_vendidas_lag6,unidades_vendidas_lag7,unidades_vendidas_mm7
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,44,4,14.0,10.0,14.0,15.0,16.0,20.0,26.0,14.833333
169,2025-11-01,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,NaN,135.00,NaN,...,44,4,13.0,13.0,14.0,12.0,15.0,19.0,27.0,14.333333
170,2025-11-01,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,NaN,86.39,NaN,...,44,4,2.0,4.0,8.0,6.0,3.0,5.0,5.0,4.666667
171,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,NaN,74.09,NaN,...,44,4,3.0,9.0,1.0,5.0,6.0,1.0,3.0,4.166667
172,2025-11-01,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,NaN,34.76,NaN,...,44,4,4.0,2.0,5.0,2.0,1.0,6.0,3.0,3.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,2025-11-30,PROD_020,Quechua MH500,Outdoor,Ropa Montaña,80,False,NaN,79.64,NaN,...,48,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
884,2025-11-30,PROD_021,Manduka PRO Yoga Mat,Wellness,Esterilla Yoga,130,True,NaN,130.00,NaN,...,48,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
885,2025-11-30,PROD_022,Gaiam Premium Yoga Block,Wellness,Bloque Yoga,20,False,NaN,20.18,NaN,...,48,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
886,2025-11-30,PROD_023,Liforme Yoga Pad,Wellness,Rodillera Yoga,35,False,NaN,34.79,NaN,...,48,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
inferencia_df.shape

(720, 37)

In [6]:
inferencia_df.fecha.unique

<bound method Series.unique of 168   2025-11-01
169   2025-11-01
170   2025-11-01
171   2025-11-01
172   2025-11-01
         ...    
883   2025-11-30
884   2025-11-30
885   2025-11-30
886   2025-11-30
887   2025-11-30
Name: fecha, Length: 720, dtype: datetime64[ns]>